In [1]:
import pandas as pd
import requests
import yfinance as yf
import os

txt_path = f'../../data/raw_data/tickers_448_v2.txt'
with open(txt_path, 'r') as f:
    tickers = f.read().splitlines()
    
# Settings
start_date = '2010-01-01'
end_date = '2025-01-01'


print(f"Downloading {len(tickers)} tickers from {start_date} to {end_date}...")

# Download daily OHLCV data
data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

# Keep only OHLCV columns and flatten into one DataFrame
ohlc_data = []
for ticker in tickers:    
    df = data[ticker][['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].copy()
    df = df.reset_index()
    df['Ticker'] = ticker
    ohlc_data.append(df)

# Combine all tickers into one DataFrame
full_df = pd.concat(ohlc_data, ignore_index=True)
full_df = full_df.sort_values(by=['Date', 'Ticker']).reset_index(drop=True)

print(f"✅ Combined dataset: {full_df['Ticker'].nunique()} tickers, {len(full_df)} rows.")

[***************       31%                       ]  139 of 448 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
[*********************100%***********************]  448 of 448 completed

1 Failed download:
['FI']: YFTzMissingError('possibly delisted; no timezone found')


✅ Combined dataset: 448 tickers, 1690752 rows.


In [2]:
full_df.dropna().to_parquet('../../data/processed_data/daily_ohlcv_448_v2.parquet', index=False)

In [ ]:
full_df

Price,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2010-01-04,22.453505,22.625179,22.267525,22.389128,19.856192,3815561.0,A
1,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.418383,493729600.0,AAPL
2,2010-01-04,26.000362,26.177889,25.870815,26.129908,18.321974,10829095.0,ABT
3,2010-01-04,7.978889,8.022222,7.972222,7.994444,7.601905,4813200.0,ACGL
4,2010-01-04,41.520000,42.200001,41.500000,42.070000,31.492170,3650100.0,ACN
...,...,...,...,...,...,...,...,...
1690747,2024-12-31,67.449997,68.059998,67.220001,67.519997,64.848953,2143800.0,XEL
1690748,2024-12-31,106.169998,107.900002,105.779999,107.570000,103.761795,12387800.0,XOM
1690749,2024-12-31,134.089996,134.789993,133.250000,134.160004,131.612030,1217100.0,YUM
1690750,2024-12-31,105.910004,106.500000,104.959999,105.629997,104.596786,683300.0,ZBH


: 

### Chunk to download tickers from given ticker list ###


## Testing Data Loading & Preprocessing ##

In [7]:
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Get the HTML content manually
response = requests.get(url, headers=headers)
response.raise_for_status()

# Parse tables from the page
sp500 = pd.read_html(response.text)[0]

# Extract and clean ticker symbols
tickers = sp500['Symbol'].str.replace('.', '-', regex=False).tolist()
print(tickers[:10])
print(f"✅ Retrieved {len(tickers)} S&P 500 tickers.")


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']
✅ Retrieved 503 S&P 500 tickers.


/tmp/ipykernel_32409/328594148.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(response.text)[0]


In [8]:
import os
print(os.getcwd())

exclude = ['MTRJF', 'PBMRF', 'BLBRF', 'ACPIF', 'PIKQF', 'UBEOF', 'AITUF', 
           'AVIJF', 'MNTHF', 'SBSNF', 'TSUSF', 'JPTXF', 'AYAAF', 'AKBLF', 
           'PORBF', 'COLZF', 'YTLCF', 'OLCLF', 'HUFAF', 'GZPHF', 'NPSCGF',
           'SNYYF', 'PTAIF', 'CHFFF', 'MAANF', 'PHJMF']




with open(f'../../data/raw_data/top_100_tickers.txt', 'r') as f:
    tickers_curr = f.readlines()
    tickers_curr = [i[:-1] for i in tickers_curr]
    
additional = [i for i in tickers if i not in tickers_curr and i not in exclude]

/home/yinkiat/fyp-kiat/ML_Core/src/utils


In [9]:
import pandas as pd
import yfinance as yf

# Settings
start_date = '2010-01-01'
end_date = '2025-01-01'

# Example: assume you already have `tickers` list
# tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']

print(f"Downloading {len(additional)} tickers from {start_date} to {end_date}...")

# Download daily OHLCV data
data = yf.download(
    additional,
    start=start_date,
    end=end_date,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

# Keep only OHLCV columns and flatten into one DataFrame
ohlc_data = []
for ticker in additional:    
    df = data[ticker][['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].copy()
    df = df.reset_index()
    df['Ticker'] = ticker
    ohlc_data.append(df)

# Combine all tickers into one DataFrame
full_df = pd.concat(ohlc_data, ignore_index=True)
full_df = full_df.sort_values(by=['Date', 'Ticker']).reset_index(drop=True)

print(f"✅ Combined dataset: {full_df['Ticker'].nunique()} tickers, {len(full_df)} rows.")

[*********************100%***********************]  479 of 479 completed


✅ Combined dataset: 479 tickers, 1807746 rows.


In [73]:
data['AAPL'][['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]

Price,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2010-01-04,7.622500,7.660714,7.585000,7.643214,6.424606,493729600
2010-01-05,7.664286,7.699643,7.616071,7.656429,6.435712,601904800
2010-01-06,7.656429,7.686786,7.526786,7.534643,6.333344,552160000
2010-01-07,7.562500,7.571429,7.466071,7.520714,6.321636,477131200
2010-01-08,7.510714,7.571429,7.466429,7.570714,6.363665,447610800
...,...,...,...,...,...,...
2024-12-24,255.490005,258.209991,255.289993,258.200012,257.286682,23234700
2024-12-26,258.190002,260.100006,257.630005,259.019989,258.103729,27237100
2024-12-27,257.829987,258.700012,253.059998,255.589996,254.685867,42355300


In [10]:
all_unique_tickers = full_df['Ticker'].unique()
filtered = []
for ticker in all_unique_tickers:
    ticker_df = full_df[full_df['Ticker'] == ticker].dropna().sort_values(by='Date')
    if ticker_df['Date'].iloc[0] > pd.to_datetime('2010-01-04') or ticker_df['Date'].iloc[-1] < pd.to_datetime('2024-12-31'):
        print(f'Dropping {ticker}, Ticker Start: {ticker_df['Date'].iloc[0]}, Ticker End: {ticker_df['Date'].iloc[-1]}')
    else:
        filtered.append(ticker)

print(f'Kept {len(filtered)} tickers')
full_df_filtered = full_df[full_df['Ticker'].isin(filtered)]

Dropping ABBV, Ticker Start: 2013-01-02 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping ABNB, Ticker Start: 2020-12-10 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping ALLE, Ticker Start: 2013-11-18 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping AMCR, Ticker Start: 2012-05-15 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping ANET, Ticker Start: 2014-06-06 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping APP, Ticker Start: 2021-04-15 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping APTV, Ticker Start: 2011-11-17 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CARR, Ticker Start: 2020-03-19 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CBOE, Ticker Start: 2010-06-15 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CDW, Ticker Start: 2013-06-27 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CEG, Ticker Start: 2022-01-19 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CFG, Ticker Start: 2014-09-24 00:00:00, Ticker End: 2024-12-31 00:00:00
Dropping CHTR, Ticke

In [11]:
curr_new_additional = full_df_filtered['Ticker'].unique().tolist()
curr_new_additional.extend(tickers_curr)


In [12]:
len(set(curr_new_additional))


504

In [13]:
# Write list of tickers to text file
with open(f'../../data/raw_data/tickers_{len(set(curr_new_additional))}_v2.txt', 'w') as f:
    for ticker in set(curr_new_additional):
        f.write(f"{ticker}\n")

### Data Download using FinnHub ###

In [7]:
import finnhub
import os
from dotenv import load_dotenv
os.chdir(rf'c:/Users/yinki/OneDrive/NUS/BT4101/fyp-kiat/ML_Core/src/')
print(os.getcwd())

load_dotenv('../config/.env')
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

# Read in the saved tickers 
top_sorted_tickers = []

with open('../data/raw_data/tickers_test_top_116.txt', "r") as f:
    for line in f:
        # split by tab and take the first part (symbol)
        symbol = line.strip().split("\t")[0]
        top_sorted_tickers.append(symbol)

print(f"Loaded {len(top_sorted_tickers)} tickers")

c:\Users\yinki\OneDrive\NUS\BT4101\fyp-kiat\ML_Core\src
Loaded 116 tickers


In [45]:
len(curr_new_additional)

539

In [1]:

with open(f'../../data/raw_data/tickers_504_v2.txt', 'r') as f:
    curr_new_additional = f.readlines()
    curr_new_additional = [i[:-1] for i in curr_new_additional]
    

In [15]:
len(curr_new_additional)

504

In [16]:
from numpy import full
import tqdm
import yfinance as yf
import pandas as pd

start_date = '2010-01-01'
end_date = '2025-01-01'

## Crawl Daily OHLC data for each ticker
all_dfs = []
not_fouund = []
for i in range(0, len(curr_new_additional), 10):
    print(f"Fetching data for {curr_new_additional[i:i+10]}")
    tickers = curr_new_additional[i:i+10]
    ohlc_data_all = {}
    try:
        ohlc_data = yf.download(
            tickers,
            start=start_date,
            end=end_date,
            interval="1d",
            auto_adjust=False,
            group_by="Ticker",
            threads=True
        )
        # keep only OHLC columns
        for ticker in tickers:
            ohlc_data_all[ticker] = ohlc_data[ticker][['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].copy()
            ohlc_data_all[ticker]['Ticker'] = ticker
            ohlc_data_all[ticker] = ohlc_data_all[ticker].reset_index()  # make Date a column
        
        full_df = pd.concat(ohlc_data_all.values(), ignore_index=True)
        full_df = full_df.sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
        all_dfs.append(full_df)
            
    except Exception as e:
        print(f"Error fetching data for {tickers}: {e}")
        not_fouund.append(tickers)
        
full_ohlc_df = pd.concat(all_dfs, ignore_index=True).sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
full_ohlc_df.to_parquet('../../data/raw_data/daily_ohlc_data.parquet', engine='fastparquet', index=False)
print(f'Total rows fetched: {len(full_ohlc_df)}')

Fetching data for ['AAPL', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADP', 'ADSK', 'AEP', 'AES']


[*********************100%***********************]  10 of 10 completed
[                       0%                       ]

Fetching data for ['AFL', 'AIG', 'AIZ', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALL', 'AMAT', 'AMD']


[*********************100%***********************]  10 of 10 completed
[**********            20%                       ]  2 of 10 completed

Fetching data for ['AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'AON', 'AOS', 'APA', 'APD', 'APH']


...

Fetching data for ['MNTHF', 'GVDBF', 'AYAAF', 'AMKAF', 'HUFAF', 'BLBRF', 'ACPIF', 'AKBLF', 'SNYYF', 'RPGRF']


[*********************100%***********************]  10 of 10 completed


Fetching data for ['RY', 'IMBBF', 'NVZMF', 'COLZF']


[*********************100%***********************]  4 of 4 completed


Total rows fetched: 1902096


In [17]:
tickers_with_na = full_ohlc_df.loc[
    full_ohlc_df[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].isna().any(axis=1),
    'Ticker'
].unique()

print(f"Number of tickers with missing data: {len(tickers_with_na)}")
print(tickers_with_na)


Number of tickers with missing data: 56
['ACPIF' 'AITUF' 'AJISF' 'ANPDF' 'APO' 'AYAAF' 'BLBRF' 'BOMXF' 'BRDCF'
 'BWLP' 'CABJF' 'CG' 'CGHOF' 'CHFFF' 'DSKIF' 'EMBVF' 'GPAEF' 'GVDBF'
 'GZPHF' 'HGKGF' 'HUFAF' 'IMBBF' 'IPXHF' 'ITOCF' 'JGCCF' 'JPTXF' 'KAOCF'
 'MDNDF' 'MNTHF' 'MTNOF' 'MTRJF' 'NHNKF' 'NOKBF' 'NSANF' 'NVZMF' 'OLCLF'
 'PHJMF' 'PIKQF' 'PORBF' 'PSX' 'PTAIF' 'PTFRF' 'RPGRF' 'SBSNF' 'SHPMF'
 'SLMAF' 'SNYYF' 'STAEF' 'TISCF' 'TSIHF' 'TSUSF' 'TTDKF' 'TTFNF' 'UBEOF'
 'XROLF' 'YTLCF']


In [18]:
# Select rows with any missing OHLCV data
rows_with_na = full_ohlc_df.loc[
    full_ohlc_df[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].isna().any(axis=1)
]

print(rows_with_na)


Price        Date  Open  High  Low  Close  Adj Close  Volume Ticker
5      2010-01-04   NaN   NaN  NaN    NaN        NaN     NaN  ACPIF
17     2010-01-04   NaN   NaN  NaN    NaN        NaN     NaN  AITUF
20     2010-01-04   NaN   NaN  NaN    NaN        NaN     NaN  AJISF
34     2010-01-04   NaN   NaN  NaN    NaN        NaN     NaN  ANPDF
40     2010-01-04   NaN   NaN  NaN    NaN        NaN     NaN    APO
...           ...   ...   ...  ...    ...        ...     ...    ...
570852 2014-07-03   NaN   NaN  NaN    NaN        NaN     NaN  NHNKF
571356 2014-07-07   NaN   NaN  NaN    NaN        NaN     NaN  NHNKF
571860 2014-07-08   NaN   NaN  NaN    NaN        NaN     NaN  NHNKF
572364 2014-07-09   NaN   NaN  NaN    NaN        NaN     NaN  NHNKF
572868 2014-07-10   NaN   NaN  NaN    NaN        NaN     NaN  NHNKF

[31963 rows x 8 columns]


In [21]:
final_tickers = [i for i in full_ohlc_df['Ticker'].unique() if i not in tickers_with_na]

In [23]:
print(len(set(final_tickers)))

448


In [24]:
# Write list of tickers to text file
with open(f'../../data/raw_data/tickers_{len(set(final_tickers))}_v2.txt', 'w') as f:
    for ticker in set(final_tickers):
        f.write(f"{ticker}\n")

In [2]:
# Crawl as much as possible to 20 years data of price for each ticker
import finnhub
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

all_ticker_data = {}

# Loop over all tickers
for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    print(f"Fetching {symbol} ...")
    try:
        end_time = int(time.time())   # current timestamp
        start_time = end_time - 365 * 24 * 3600  # 1 year window in seconds

        dfs = []  # store chunks here

        while True:
            res = finnhub_client.stock_candles(symbol, 'D', start_time, end_time)

            if res['s'] != 'ok' or len(res['t']) == 0:
                break  # no more data

            df = pd.DataFrame(res)

            # Convert timestamp to date
            df['Date'] = pd.to_datetime(df['t'], unit='s').dt.strftime('%Y-%m-%d')

            df = df.rename(columns={
                'o': 'Open',
                'h': 'High',
                'l': 'Low',
                'c': 'Close',
                'v': 'Volume'
            })
            df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]

            dfs.append(df)

            # Shift backwards: new end_time = oldest timestamp - 1
            end_time = res['t'][0] - 1
            start_time = end_time - 365 * 24 * 3600  # move back another year

            # Prevent too many requests per minute
            time.sleep(0.5)

        # Merge all chunks
        if dfs:
            all_data = pd.concat(dfs, ignore_index=True)
            all_data.drop_duplicates(subset=['Date'], inplace=True)
            all_data.sort_values('Date', inplace=True)
            all_data.to_parquet(f'../data/raw_data/{symbol}_daily.parquet', index=False)
            #all_ticker_data[symbol] = all_data

    except Exception as e:
        print(f"Error fetching {symbol}: {e}")
        continue




Tickers:   0%|          | 0/1000 [00:00<?, ?it/s]

Fetching AMMNF ...


Tickers:   0%|          | 1/1000 [00:00<14:37,  1.14it/s]

Error fetching AMMNF: FinnhubAPIException(status_code: 403): You don't have access to this resource.
Fetching PTAIF ...


Tickers:   0%|          | 2/1000 [00:01<08:38,  1.93it/s]

...

Error fetching PIKQF: FinnhubAPIException(status_code: 403): You don't have access to this resource.
Fetching CAMDF ...


Tickers:   1%|          | 8/1000 [00:02<04:39,  3.56it/s]

Error fetching CAMDF: FinnhubAPIException(status_code: 403): You don't have access to this resource.
Fetching PKNPF ...


Tickers:   1%|          | 8/1000 [00:03<06:12,  2.66it/s]


KeyboardInterrupt: 

In [8]:
# Insider Transactions Data
all_insider_data = []

# Define global date range
global_start = datetime(2005, 9, 17)
global_end = datetime(2025, 9, 17)

# Loop over tickers
for symbol in top_sorted_tickers:
    print(f"Fetching insider transactions for {symbol} ...")
    try:
        # Step backwards year by year
        end_date = global_end
        with tqdm(desc=f"{symbol} date ranges", leave=False) as pbar:
            while end_date > global_start:
                start_date = end_date - timedelta(days=365)

                try:
                    res = finnhub_client.stock_insider_transactions(
                        symbol, 
                        _from=start_date.strftime("%Y-%m-%d"), 
                        to=end_date.strftime("%Y-%m-%d")
                    )

                    if res and "data" in res and len(res["data"]) > 0:
                        df = pd.DataFrame(res["data"])
                        df["symbol"] = symbol
                        all_insider_data.append(df)

                except Exception as e:
                    print(f"Error fetching insider transactions for {symbol} ({start_date}–{end_date}): {e}")

                # Shift window backwards
                end_date = start_date
                time.sleep(0.5)  # respect rate limits
                pbar.update(1)
    except Exception as e:
        print(f"Error with ticker {symbol}: {e}")
        continue

# Merge all into a single DataFrame
if all_insider_data:
    insider_df = pd.concat(all_insider_data, ignore_index=True)
    print(insider_df.head())
    insider_df.to_parquet("../data/raw_data/insider_transactions.parquet", index=False)


Fetching insider transactions for AMMNF ...


Fetching insider transactions for PTAIF ...


Fetching insider transactions for PHJMF ...


Fetching insider transactions for PBMRF ...


Fetching insider transactions for ADARF ...


...

Fetching insider transactions for AHEXF ...


Fetching insider transactions for ADMA ...


Fetching insider transactions for PMOIF ...


Fetching insider transactions for EEFT ...


KeyboardInterrupt: 

In [ ]:
import finnhub
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

all_insider_sentiment = []

# Define global date range
global_start = datetime(2005, 9, 17)
global_end = datetime(2025, 9, 17)

# Outer loop over tickers
for symbol in tqdm(tickers, desc="Tickers"):
    try:
        end_date = global_end

        # Inner loop for date ranges
        with tqdm(desc=f"{symbol} sentiment ranges", leave=False) as pbar:
            while end_date > global_start:
                start_date = end_date - timedelta(days=365)

                try:
                    res = finnhub_client.stock_insider_sentiment(
                        symbol,
                        _from=start_date.strftime("%Y-%m-%d"),
                        to=end_date.strftime("%Y-%m-%d")
                    )

                    if res and "data" in res and len(res["data"]) > 0:
                        df = pd.DataFrame(res["data"])
                        df["symbol"] = symbol  # reinforce symbol column
                        all_insider_sentiment.append(df)

                except Exception as e:
                    print(f"Error fetching insider sentiment for {symbol} ({start_date}–{end_date}): {e}")

                # Move window backwards
                end_date = start_date
                time.sleep(0.5)  # respect rate limits
                pbar.update(1)

    except Exception as e:
        print(f"Error with {symbol}: {e}")
        continue

# Merge all into a single DataFrame
if all_insider_sentiment:
    sentiment_df = pd.concat(all_insider_sentiment, ignore_index=True)
    print(sentiment_df.head())
    sentiment_df.to_parquet("../data/raw_data/insider_sentiment.parquet", index=False)


In [15]:
### Company Financials ###
# Statements you want (balance sheet, income statement, cash flow)
import tqdm
import time

statements = ["bs", "ic", "cf"]
statements_dicts = {s: [] for s in statements}

# Loop over tickers
for symbol in tqdm.tqdm(['AAPL'], desc="Tickers"):
    try:
        # Loop over statements
        for stmt in tqdm.tqdm(statements, desc=f"{symbol} statements", leave=False):
            try:
                res = finnhub_client.financials(symbol, stmt, "quarterly")
                if res and "financials" in res and len(res["financials"]) > 0:
                    df = pd.DataFrame(res["financials"])
                    df["symbol"] = symbol
                    df["statement_type"] = stmt
                    statements_dicts[stmt].append(df)

            except Exception as e:
                print(f"Error fetching {stmt} for {symbol}: {e}")

            time.sleep(0.5)  # respect rate limits

    except Exception as e:
        print(f"Error with {symbol}: {e}")
        continue

# Merge into one DataFrame
for stmt, dfs in statements_dicts.items():
    if dfs:
        stmt_df = pd.concat(dfs, ignore_index=True)
        stmt_df.to_parquet(f"../data/raw_data/financials_{stmt}_quarterly_all_tickers.parquet", index=False)
    else:
        print(f"No data for statement type {stmt}")

Tickers: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it]


In [17]:
stmt_df.columns


Index(['capex', 'cashDividendsPaid', 'cashTaxesPaid', 'changeinCash',
       'changesinWorkingCapital', 'depreciationAmortization', 'fcf',
       'issuanceReductionCapitalStock', 'issuanceReductionDebtNet',
       'netCashFinancingActivities', 'netIncomeStartingLine',
       'netInvestingCashFlow', 'netOperatingCashFlow',
       'otherFundsFinancingItems', 'otherFundsNonCashItems',
       'otherInvestingCashFlowItemsTotal', 'period', 'stockBasedCompensation',
       'cashInterestPaid', 'deferredTaxesInvestmentTaxCredit',
       'foreignExchangeEffects', 'symbol', 'statement_type'],
      dtype='object')

In [ ]:
#### Historical Market Cap ###
import finnhub
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

all_market_caps = {}  # dictionary to store per ticker

# Define global date range
global_start = datetime(2005, 9, 17)
global_end = datetime(2025, 9, 17)

for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    dfs = []
    end_date = global_end

    with tqdm(desc=f"{symbol} market cap ranges", leave=False) as pbar:
        while end_date > global_start:
            start_date = end_date - timedelta(days=365)  # 1-year chunks

            try:
                res = finnhub_client.historical_market_cap(
                    symbol,
                    _from=start_date.strftime("%Y-%m-%d"),
                    to=end_date.strftime("%Y-%m-%d")
                )

                if res and "data" in res and len(res["data"]) > 0:
                    df = pd.DataFrame(res["data"])
                    df["symbol"] = symbol
                    dfs.append(df)

            except Exception as e:
                print(f"Error fetching market cap for {symbol} ({start_date}-{end_date}): {e}")

            # Move window backwards
            end_date = start_date
            time.sleep(0.5)  # respect rate limits
            pbar.update(1)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df.drop_duplicates(subset=["atDate"], inplace=True)
        full_df.sort_values("atDate", inplace=True)
        all_market_caps[symbol] = full_df

# Optional: merge all tickers into one DataFrame
if all_market_caps:
    market_cap_df = pd.concat(all_market_caps.values(), ignore_index=True)
    market_cap_df.to_parquet("../data/raw_data/market_cap_history.parquet", index=False)


In [ ]:
### Historical Employee Count ###
all_employee_counts = {}  # dictionary to store per ticker

# Define global date range
global_start = datetime(2005, 9, 17)
global_end = datetime(2025, 9, 17)

for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    dfs = []
    end_date = global_end

    with tqdm(desc=f"{symbol} employee count ranges", leave=False) as pbar:
        while end_date > global_start:
            start_date = end_date - timedelta(days=365)  # 1-year chunks

            try:
                res = finnhub_client.historical_employee_count(
                    symbol,
                    _from=start_date.strftime("%Y-%m-%d"),
                    to=end_date.strftime("%Y-%m-%d")
                )

                if res and "data" in res and len(res["data"]) > 0:
                    df = pd.DataFrame(res["data"])
                    df["symbol"] = symbol
                    dfs.append(df)

            except Exception as e:
                print(f"Error fetching employee count for {symbol} ({start_date}-{end_date}): {e}")

            # Move window backwards
            end_date = start_date
            time.sleep(0.5)  # respect API limits
            pbar.update(1)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df.drop_duplicates(subset=["atDate"], inplace=True)
        full_df.sort_values("atDate", inplace=True)
        all_employee_counts[symbol] = full_df


# Optional: merge all tickers into one DataFrame
if all_employee_counts:
    employee_df = pd.concat(all_employee_counts.values(), ignore_index=True)
    employee_df.to_parquet("../data/raw_data/employee_count_history.parquet", index=False)


In [ ]:
### Stock Upgrading and Downgrading ###
all_upgrades = {}  # dictionary per ticker

# Define global date range
global_start = datetime(2000, 1, 1)
global_end = datetime(2025, 9, 17)

for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    dfs = []
    end_date = global_end

    with tqdm(desc=f"{symbol} upgrades/downgrades", leave=False) as pbar:
        while end_date > global_start:
            start_date = end_date - timedelta(days=365)  # 1-year chunks
            try:
                res = finnhub_client.upgrade_downgrade(
                    symbol=symbol,
                    _from=start_date.strftime("%Y-%m-%d"),
                    to=end_date.strftime("%Y-%m-%d")
                )

                if res and len(res) > 0:
                    df = pd.DataFrame(res)
                    # Convert gradeTime to datetime string with seconds
                    df['gradeTime'] = pd.to_datetime(df['gradeTime'], unit='s').dt.strftime('%Y-%m-%d %H:%M:%S')
                    df['symbol'] = symbol
                    dfs.append(df)

            except Exception as e:
                print(f"Error fetching upgrade/downgrade for {symbol} ({start_date}-{end_date}): {e}")

            # Shift window backwards
            end_date = start_date
            time.sleep(0.5)  # respect rate limits
            pbar.update(1)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df.drop_duplicates(subset=['gradeTime', 'company'], inplace=True)
        full_df.sort_values('gradeTime', inplace=True)
        all_upgrades[symbol] = full_df

# Optional: merge all tickers into one DataFrame
if all_upgrades:
    upgrades_df = pd.concat(all_upgrades.values(), ignore_index=True)
    upgrades_df.to_parquet("../data/raw_data/stock_upgrade_downgrade.parquet", index=False)


In [ ]:
### Earnings Surprises ###
all_earnings = {}  # dictionary per ticker

# Define global date range
global_start = datetime(2000, 1, 1)
global_end = datetime(2025, 9, 17)

for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    dfs = []
    end_date = global_end

    with tqdm(desc=f"{symbol} earnings ranges", leave=False) as pbar:
        while end_date > global_start:
            start_date = end_date - timedelta(days=365*2)  # 2-year chunks (quarterly data is sparse)

            try:
                res = finnhub_client.company_earnings(
                    symbol,
                    limit=None  # get full history within date range
                )

                if res and len(res) > 0:
                    df = pd.DataFrame(res)
                    df['symbol'] = symbol
                    # Ensure period column is datetime
                    df['period'] = pd.to_datetime(df['period']).dt.strftime('%Y-%m-%d')
                    dfs.append(df)
                else:
                    break  # no more data

            except Exception as e:
                print(f"Error fetching earnings for {symbol} ({start_date}-{end_date}): {e}")

            # Move window backwards
            end_date = start_date
            time.sleep(0.5)
            pbar.update(1)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df.drop_duplicates(subset=['period'], inplace=True)
        full_df.sort_values('period', inplace=True)
        all_earnings[symbol] = full_df


# Optional: merge all tickers into one DataFrame
if all_earnings:
    earnings_df = pd.concat(all_earnings.values(), ignore_index=True).drop_duplicates().reset_index(drop=True)
    earnings_df.to_parquet("../data/raw_data/earnings_surprises.parquet", index=False)


In [ ]:
### USA Spending ###
all_usa_spending = {}  # dictionary per ticker

# Define global date range
global_start = datetime(2005, 1, 1)
global_end = datetime(2025, 9, 17)

for symbol in tqdm(top_sorted_tickers, desc="Tickers"):
    dfs = []
    end_date = global_end

    with tqdm(desc=f"{symbol} USA Spending ranges", leave=False) as pbar:
        while end_date > global_start:
            start_date = end_date - timedelta(days=365)  # 1-year chunks

            try:
                res = finnhub_client.stock_usa_spending(
                    symbol=symbol,
                    _from=start_date.strftime("%Y-%m-%d"),
                    to=end_date.strftime("%Y-%m-%d")
                )

                if res and "data" in res and len(res["data"]) > 0:
                    df = pd.DataFrame(res["data"])
                    # Convert date columns to YYYY-MM-DD strings
                    for col in ["actionDate", "performanceStartDate", "performanceEndDate"]:
                        if col in df.columns:
                            df[col] = pd.to_datetime(df[col]).dt.strftime('%Y-%m-%d')
                    df["symbol"] = symbol
                    dfs.append(df)

            except Exception as e:
                print(f"Error fetching USA Spending for {symbol} ({start_date}-{end_date}): {e}")

            # Shift window backwards
            end_date = start_date
            time.sleep(0.5)  # respect API limits
            pbar.update(1)

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        full_df.drop_duplicates(subset=["actionDate", "awardDescription"], inplace=True)
        full_df.sort_values("actionDate", inplace=True)
        all_usa_spending[symbol] = full_df


# Optional: merge all tickers into one DataFrame
if all_usa_spending:
    usa_spending_df = pd.concat(all_usa_spending.values(), ignore_index=True)
    usa_spending_df.to_parquet("../data/raw_data/usa_spending_history.parquet", index=False)


### Looking at pulled data ###

In [33]:
import os 
import pandas as pd

tickers = []
for files in os.listdir('../data/raw_data/'):
    if files.endswith('.parquet') and files :
        df = pd.read_parquet(os.path.join('../data/raw_data/', files), engine='fastparquet')
        print(f"File: {files}, Rows: {len(df)}, Number of Symbols: {df['symbol'].nunique() if 'symbol' in df.columns else 'N/A'}")
        if 'symbol' in df.columns:
            tickers.append(df['symbol'].unique().tolist())
        else:
            continue


# Get the smallest ticker list subset as those with symbol column thats present in all the lists
common_tickers = set(tickers[0])
for t in tickers[1:]:
    common_tickers = common_tickers.intersection(set(t))
    
common_tickers = sorted(list(common_tickers))
print(f"Common tickers across all datasets: {len(common_tickers)}")

File: congressional_trading_disclosure.parquet, Rows: 39038, Number of Symbols: 121
File: daily_ohlc_data.parquet, Rows: 5088850, Number of Symbols: N/A
File: earnings_quality_score.parquet, Rows: 28526, Number of Symbols: 432
File: earnings_surprises.parquet, Rows: 2484, Number of Symbols: 655
File: employee_count_history.parquet, Rows: 657, Number of Symbols: 63
File: financials_bs_quarterly_all_tickers.parquet, Rows: 53997, Number of Symbols: 974
File: financials_cf_quarterly_all_tickers.parquet, Rows: 53076, Number of Symbols: 969
File: financials_ic_quarterly_all_tickers.parquet, Rows: 55368, Number of Symbols: 974
File: historical_esg_scores.parquet, Rows: 316, Number of Symbols: 48
File: insider_sentiment.parquet, Rows: 33567, Number of Symbols: 350
File: insider_transactions.parquet, Rows: 398473, Number of Symbols: 635
File: market_cap_history.parquet, Rows: 6200, Number of Symbols: 11
File: social_sentiment.parquet, Rows: 26247, Number of Symbols: 330
File: test_raw_data.parq